# Project 9: Hospital Appointment Operations Analysis
**Industry:** Healthcare Operations  
**Dataset:** `09_hospital_appointments.csv`  
**Tasks:** 124 to 138

**Scenario:** Analyze appointment attendance, waiting times and department workload to identify operational bottlenecks.

> Note: This is an *operational* dataset, not a clinical one. No medical conclusions are drawn.

## Step 0: Setup and upload the dataset

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)

# In Google Colab: choose 09_hospital_appointments.csv when the upload button appears.
# Outside Colab (or if the file is already in the folder) it is read directly.
import os
if not os.path.exists("09_hospital_appointments.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError("Place 09_hospital_appointments.csv next to this notebook.")

## Task 124: Load and inspect appointment records

In [ ]:
df = pd.read_csv("09_hospital_appointments.csv")
print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").round(2)

In [ ]:
# Convert date column to datetime
df["Appointment_Date"] = pd.to_datetime(df["Appointment_Date"], errors="coerce")
print(df.dtypes)

**Observation:** The dataset has 8 columns covering appointment ID, date, department, appointment type, attendance status, waiting time, SMS reminder flag and consultation fee.

## Task 125: Identify and handle missing waiting times

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
missing_wait = df["Waiting_Time_Minutes"].isna().sum()
print(f"\nMissing waiting times: {missing_wait} ({missing_wait/len(df)*100:.2f}%)")

In [ ]:
# Handling method: fill with the MEDIAN waiting time of the same department.
# Median is used because it is not distorted by extreme waits.
df["Waiting_Time_Minutes"] = df.groupby("Department")["Waiting_Time_Minutes"] \
    .transform(lambda s: s.fillna(s.median()))
print("Missing waiting times after handling:", df["Waiting_Time_Minutes"].isna().sum())

**Observation:** Missing waits were filled with the department median so that no rows are lost and department comparisons stay fair.

## Task 126: Check duplicate Appointment_ID values

In [ ]:
dups = df["Appointment_ID"].duplicated().sum()
print("Duplicate Appointment_ID rows:", dups)
print("Fully identical duplicate rows:", df.duplicated().sum())
df[df["Appointment_ID"].duplicated(keep=False)].sort_values("Appointment_ID").head(10)

In [ ]:
df = df.drop_duplicates(subset="Appointment_ID", keep="first").reset_index(drop=True)
print("Rows after removing duplicates:", len(df))

**Observation:** Duplicate IDs would inflate appointment counts and fees, so only the first record of each ID was kept.

## Task 127: Total appointments and attendance-status percentages

In [ ]:
print("Total appointments:", len(df))
status = pd.DataFrame({
    "Count": df["Attendance_Status"].value_counts(),
    "Percent": (df["Attendance_Status"].value_counts(normalize=True) * 100).round(2)
})
status

## Task 128: Average waiting time by department

In [ ]:
wait_dept = df.groupby("Department").agg(
    Avg_Wait_Min=("Waiting_Time_Minutes", "mean"),
    Appointments=("Appointment_ID", "count")
).round(1).sort_values("Avg_Wait_Min", ascending=False)
wait_dept

## Task 129: Departments with the highest no-show percentage

In [ ]:
df["Is_NoShow"] = (df["Attendance_Status"] == "No-Show").astype(int)

noshow = df.groupby("Department").agg(
    Appointments=("Appointment_ID", "count"),
    NoShows=("Is_NoShow", "sum"),
    NoShow_Pct=("Is_NoShow", lambda s: s.mean() * 100)
).round(2).sort_values("NoShow_Pct", ascending=False)
noshow

In [ ]:
print("Highest no-show department:", noshow.index[0], f"({noshow.iloc[0]['NoShow_Pct']}%)")

## Task 130: Compare no-show rates for SMS reminder Yes and No

In [ ]:
pd.crosstab(df["SMS_Reminder"], df["Attendance_Status"], margins=True)

In [ ]:
(pd.crosstab(df["SMS_Reminder"], df["Attendance_Status"], normalize="index") * 100).round(2)

In [ ]:
sms = df.groupby("SMS_Reminder")["Is_NoShow"].mean() * 100
print(sms.round(2))
print(f"\nDifference (No reminder - Reminder): {sms['No'] - sms['Yes']:.2f} percentage points")

**Observation:** Compare the two no-show rates above. A lower rate for "Yes" suggests reminders are associated with better attendance (this shows association, not proof of cause).

## Task 131: Average consultation fee by department

In [ ]:
df.groupby("Department")["Consultation_Fee"].mean().round(2).sort_values(ascending=False)

## Task 132: Appointment volume by date and department

In [ ]:
daily = df.groupby("Appointment_Date").size()
print("Average appointments per day:", round(daily.mean(), 1))
print("\nBusiest 5 days:\n", daily.sort_values(ascending=False).head())
print("\nQuietest 5 days:\n", daily.sort_values().head())

In [ ]:
pivot = df.pivot_table(index="Appointment_Date", columns="Department",
                       values="Appointment_ID", aggfunc="count", fill_value=0)
pivot.head(10)

In [ ]:
print("Average daily workload per department:")
pivot.mean().round(1).sort_values(ascending=False)

In [ ]:
df["Weekday"] = df["Appointment_Date"].dt.day_name()
df["Weekday"].value_counts()

## Task 133: Compare waiting times for new and follow-up appointments

In [ ]:
df.groupby("Appointment_Type")["Waiting_Time_Minutes"].agg(["count", "mean", "median", "std"]).round(1)

## Task 134: NumPy median, quartiles and standard deviation of waiting time

In [ ]:
w = df["Waiting_Time_Minutes"].to_numpy()
q1, med, q3 = np.percentile(w, [25, 50, 75])

print(f"Median  : {med:.2f}")
print(f"Q1      : {q1:.2f}")
print(f"Q3      : {q3:.2f}")
print(f"Std dev : {np.std(w, ddof=1):.2f}")
print(f"Mean    : {np.mean(w):.2f}")
print(f"Min/Max : {w.min():.0f} / {w.max():.0f}")

## Task 135: Identify unusually long waiting times using IQR

In [ ]:
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(f"IQR = {iqr:.2f} | Lower bound = {lower:.2f} | Upper bound = {upper:.2f}")

outliers = df[(df["Waiting_Time_Minutes"] > upper) | (df["Waiting_Time_Minutes"] < lower)]
print(f"Outlier appointments: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

In [ ]:
outliers["Department"].value_counts()

In [ ]:
outliers.sort_values("Waiting_Time_Minutes", ascending=False)[
    ["Appointment_ID", "Department", "Appointment_Type", "Waiting_Time_Minutes"]].head()

## Task 136: Create waiting-time categories

In [ ]:
bins = [-1, 15, 30, 60, np.inf]
labels = ["Short (0-15)", "Moderate (16-30)", "Long (31-60)", "Very Long (60+)"]
df["Wait_Category"] = pd.cut(df["Waiting_Time_Minutes"], bins=bins, labels=labels)

cat = pd.DataFrame({
    "Count": df["Wait_Category"].value_counts().sort_index(),
    "Percent": (df["Wait_Category"].value_counts(normalize=True).sort_index() * 100).round(2)
})
cat

In [ ]:
(pd.crosstab(df["Department"], df["Wait_Category"], normalize="index") * 100).round(1)

## Task 137: Estimated total consultation fees for attended appointments

In [ ]:
att = df[df["Attendance_Status"] == "Attended"]
print(f"Attended appointments: {len(att)}")
print(f"Estimated total fees : {att['Consultation_Fee'].sum():,.0f}")

In [ ]:
att.groupby("Department")["Consultation_Fee"].sum().sort_values(ascending=False)

In [ ]:
lost = df[df["Attendance_Status"] == "No-Show"]["Consultation_Fee"].sum()
print(f"Potential fees lost to no-shows: {lost:,.0f}")

## Task 138: Five operational recommendations

In [ ]:
top_ns = noshow.index[0]
top_wait = wait_dept.index[0]

recs = [
    f"1. Expand SMS reminders: no-show rate is {sms['Yes']:.1f}% with reminders vs {sms['No']:.1f}% without. Send reminders to every patient.",
    f"2. Target {top_ns} (highest no-show rate, {noshow.iloc[0]['NoShow_Pct']}%) with confirmation calls and a small controlled overbooking buffer.",
    f"3. Reduce waits in {top_wait} (average {wait_dept.iloc[0]['Avg_Wait_Min']} min) by adjusting schedules or adding clinic sessions/staff.",
    f"4. Review the {len(outliers)} extreme-wait appointments (above {upper:.0f} min) to find causes such as late starts or overlapping bookings.",
    f"5. Rebalance capacity to match peak days (busiest day has {daily.max()} appointments vs average {daily.mean():.0f}); offer freed no-show slots to waitlisted patients.",
]
for r in recs:
    print(r)

## Conclusion
This analysis looked at attendance, waiting times, department workload and fees to find operational bottlenecks. The main areas to act on are no-show reduction (especially through SMS reminders), long waits in the busiest specialist departments, extreme-wait cases and uneven daily workload.

*All findings are operational only; no medical or clinical conclusions are drawn.*